# P4-A3 — Multi-Agent: the Supervisor Pattern

Instead of one agent with all the tools, the **supervisor** pattern uses a manager agent that **routes** each step to a specialist worker. Here: a `researcher` (Wikipedia) and a `mathematician` (calculator). The supervisor reads the conversation, decides who should act next — or that the task is done (`FINISH`).

Why bother? Specialists with focused tools/prompts are more reliable than one agent juggling everything — and the architecture scales: add a new capability = add a worker, not bloat an existing one.

The graph is a hub-and-spoke: `supervisor → worker → supervisor → ... → END`.

In [ ]:
# --- Setup (given) ---
import ast, operator
from typing import Literal
from pydantic import BaseModel
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import AIMessage
from langchain.agents import create_agent
from langgraph.graph import StateGraph, START, END, MessagesState

load_dotenv()
llm = ChatOpenAI(model='gpt-4o-mini')

@tool
def wikipedia_search(query: str) -> str:
    """Search Wikipedia for a short factual summary."""
    import requests
    h = {'User-Agent': 'ai-learning/0.1'}
    s = requests.get('https://en.wikipedia.org/w/api.php', params={'action':'query','list':'search','srsearch':query,'format':'json','srlimit':1}, headers=h, timeout=10).json()
    hits = s.get('query',{}).get('search',[])
    if not hits:
        return f"No article for '{query}'."
    title = hits[0]['title']
    sm = requests.get(f'https://en.wikipedia.org/api/rest_v1/page/summary/{title}', headers=h, timeout=10).json()
    return f"{title}: {sm.get('extract','')}"

_OPS = {ast.Add:operator.add, ast.Sub:operator.sub, ast.Mult:operator.mul, ast.Div:operator.truediv, ast.Pow:operator.pow, ast.USub:operator.neg}
def _ev(n):
    if isinstance(n, ast.Constant) and isinstance(n.value,(int,float)) and not isinstance(n.value,bool): return n.value
    if isinstance(n, ast.BinOp) and type(n.op) in _OPS: return _OPS[type(n.op)](_ev(n.left), _ev(n.right))
    if isinstance(n, ast.UnaryOp) and type(n.op) in _OPS: return _OPS[type(n.op)](_ev(n.operand))
    raise ValueError('bad expr')
@tool
def calculator(expression: str) -> str:
    """Evaluate an arithmetic expression safely."""
    try: return str(_ev(ast.parse(expression, mode='eval').body))
    except Exception: return f"Error: invalid expression '{expression}'."

# Two specialist worker agents (each a prebuilt agent with ONE focused tool)
researcher = create_agent(llm, [wikipedia_search])
mathematician = create_agent(llm, [calculator])

# Supervisor: an LLM router that picks who acts next (structured output -> one of these)
class Route(BaseModel):
    next: Literal['researcher', 'mathematician', 'FINISH']

class State(MessagesState):
    next: str

def supervisor(state):
    sys = ("You are a supervisor managing two workers: 'researcher' (looks up facts on Wikipedia) "
           "and 'mathematician' (does calculations). Given the conversation so far, choose who should "
           "act NEXT to make progress. Reply FINISH only when the user's request is FULLY answered.")
    decision = llm.with_structured_output(Route).invoke([{'role':'system','content':sys}] + state['messages'])
    print(f"  [supervisor] -> {decision.next}")
    return {'next': decision.next}

def make_worker_node(name, agent):
    """Wrap a worker sub-agent as a node that adds its result back to the shared messages."""
    def node(state):
        result = agent.invoke({'messages': state['messages']})
        return {'messages': [AIMessage(content=result['messages'][-1].content, name=name)]}
    return node

def route(state):
    """Send to the chosen worker, or END when the supervisor says FINISH."""
    return state['next'] if state['next'] != 'FINISH' else END

print('ready — workers:', 'researcher, mathematician')

## Task 1 — Wire the supervisor graph
Build the hub-and-spoke graph (you have all the pieces above):
- nodes: `supervisor`, `researcher` (= `make_worker_node('researcher', researcher)`), `mathematician`
- `START -> supervisor`
- a **conditional edge** from `supervisor` using `route`, mapping to the two workers and `END`
- each worker edges **back to** `supervisor`
- compile to `app`
<br>Hint: `g.add_conditional_edges('supervisor', route, {'researcher':'researcher','mathematician':'mathematician', END: END})`.

In [ ]:
# TODO: build the StateGraph(State) with supervisor + 2 workers + routing; compile to `app`


## Task 2 — A query that needs BOTH specialists
Run `app` on something like *"What is the population of Japan, and what is that number divided by 1000?"* — watch the `[supervisor] -> ...` trace route to `researcher` (get population), back to supervisor, to `mathematician` (divide), back, then `FINISH`.
<br>Print the final answer. Pass `config={'recursion_limit': 12}` to `invoke` as a safety cap (a router can loop).
<br>Hint: `app.invoke({'messages':[{'role':'user','content': ...}]}, config={'recursion_limit': 12})`.

In [ ]:
# TODO: run the two-specialist query; print the routing trace + final answer


## Task 3 — A query that needs only ONE
Run a pure-facts question (e.g. *"Who painted the Mona Lisa?"*). Show the supervisor routes only to `researcher`, then `FINISH` — the routing adapts to the task; the mathematician never runs.
<br>Goal: see that the supervisor makes dynamic decisions, not a fixed script.

In [ ]:
# TODO: run a single-specialist query; show only the researcher is used


## Task 4 — Explain (in your own words)
1. What does the supervisor pattern buy you over a single agent holding all the tools? When is it *not* worth the extra complexity?
2. The supervisor is itself just an LLM call returning structured output (`Route`). What are the failure modes of routing this way, and how would you guard against them (think back to your `recursion_limit` and to evals)?

> _your answer here_